# London and Bristol: exploratory analysis and data quality

## Findings and remaining uncertainty

Verified local execution on 9 November 2026: London contains 91,954 listing snapshots and Bristol 2,932. Positive advertised-price medians are £175 and £130. Price is missing for 34.85% and 22.89% of rows. Numeric bathrooms, instant booking and neighbourhood-group labels are entirely missing; textual bathroom descriptions permit bathroom recovery. No duplicate listing-snapshot keys were detected. These results apply to the pinned sources; rerun all cells after changing inputs.

The final summary cell displays current results and exports aggregate evidence. Missing-price mechanisms, extreme-price validity and out-of-city transfer remain unresolved. Databricks execution status is recorded separately in `docs/setup_databricks.md`.

The target is the nightly advertised `price` in GBP on the snapshot date. One row represents one listing snapshot. Raw inputs remain unchanged; quality flags are recommendations for the ETL owner. No production ETL, model fitting, or City C ingestion runs here.

The published snapshot labels are five days apart; actual listing scrape dates occur later and are audited separately. Observed city differences are descriptive and do not establish causal effects. Price quotations, revenue estimates and review value scores require explicit leakage controls. Historical and current attributes cannot support claims about future prices without temporal validation.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Local paths and Databricks volume paths share the same manifest structure.
IN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ or 'dbutils' in globals()
if IN_DATABRICKS:
    dbutils.widgets.text('input_root', '/Volumes/workspace/airbnb_eda/raw')
    dbutils.widgets.text('output_root', '/Volumes/workspace/airbnb_eda/artifacts')
    INPUT_ROOT = Path(dbutils.widgets.get('input_root'))
    OUTPUT_ROOT = Path(dbutils.widgets.get('output_root'))
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'sources.json').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    INPUT_ROOT = Path(os.environ.get('AIRBNB_INPUT_ROOT', str(PROJECT_ROOT)))
    OUTPUT_ROOT = Path(os.environ.get('AIRBNB_OUTPUT_ROOT', str(PROJECT_ROOT / 'docs')))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
FIGURE_ROOT.mkdir(exist_ok=True)
manifest = json.loads((INPUT_ROOT / 'sources.json').read_text())
sources = manifest['sources']
if len(sources) != 2 or {s['city'] for s in sources} != {'london', 'bristol'}:
    raise ValueError('EDA accepts exactly the London and Bristol source records.')
for s in sources:
    expected = f"https://data.insideairbnb.com/united-kingdom/england/{s['city']}/{s['snapshot_date']}/data/listings.csv.gz"
    if s['url'] != expected or s['relative_path'] != f"data/raw/{s['city']}/{s['snapshot_date']}/listings.csv.gz":
        raise ValueError('Unexpected source URL or input path.')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_rows', 100)

def table(frame):
    """Render aggregate tables without optional formatting dependencies."""
    f = frame.copy().astype(object).where(frame.notna(), 'NA')
    def clean(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    return '\n'.join(['| ' + ' | '.join(map(clean, f.columns)) + ' |',
                      '| ' + ' | '.join(['---'] * len(f.columns)) + ' |'] +
                     ['| ' + ' | '.join(map(clean, row)) + ' |' for row in f.itertuples(index=False, name=None)])

def show(frame):
    display(Markdown(table(frame)))


## Provenance and input integrity

Files must match the recorded SHA-256 checksums. The manifest pins the latest published detailed snapshots verified on its recorded verification date. Refresh sources deliberately before a new study; do not silently replace a frozen snapshot. Names, URLs, descriptions, host identifiers and raw quotation payloads are not loaded into the analytical frame or displayed.


In [ ]:
CORE = ['id', 'price', 'room_type', 'accommodates', 'property_type', 'neighbourhood_cleansed']
NUMERIC = ['accommodates', 'bathrooms', 'bedrooms', 'beds', 'latitude', 'longitude',
           'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60',
           'availability_90', 'availability_365', 'availability_eoy', 'number_of_reviews',
           'number_of_reviews_ltm', 'number_of_reviews_l30d', 'reviews_per_month',
           'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
           'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
           'review_scores_value', 'estimated_occupancy_l365d']
CATEGORY = ['room_type', 'property_type', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
            'host_is_superhost', 'instant_bookable', 'bathrooms_text', 'has_availability']
MONETARY = ['price', 'estimated_revenue_l365d', 'price_quote_total_price', 'price_quote_price_per_night']
used = ['id', 'last_scraped'] + NUMERIC + CATEGORY + MONETARY
used = list(dict.fromkeys(used))
frames, provenance, headers = [], [], {}
for s in sources:
    path = INPUT_ROOT / s['relative_path']
    with path.open('rb') as handle:
        digest = hashlib.file_digest(handle, 'sha256').hexdigest()
    if digest != s['sha256']:
        raise ValueError(f"Input checksum mismatch: {s['city']}")
    header = pd.read_csv(path, nrows=0).columns.tolist()
    headers[s['city']] = header
    absent = sorted(set(CORE) - set(header))
    if absent:
        raise ValueError(f"Required columns absent in {s['city']}: {absent}")
    frame = pd.read_csv(path, dtype='string', usecols=lambda c: c in used, keep_default_na=True)
    frame = frame.reindex(columns=used)
    frame['city'] = s['city']
    frame['snapshot_date'] = s['snapshot_date']
    frames.append(frame)
    provenance.append({'city': s['city'], 'snapshot_date': s['snapshot_date'], 'rows': len(frame),
                       'source_columns': len(header), 'loaded_columns': len(frame.columns) - 2,
                       'sha256_verified': True, 'source_url': s['url']})
raw = pd.concat(frames, ignore_index=True)
provenance = pd.DataFrame(provenance)
show(provenance)


## Schema, grain and missingness

Source CSV fields are read as nullable strings to preserve formatting. Numeric conversion is audited separately. Blank strings are missing. Missingness rates use every source row as the denominator. Pattern frequencies include complete rows; review-score missingness is compared with review history to distinguish structural absence from unexplained gaps. Optional columns absent in a snapshot are reported rather than silently assumed present.


In [ ]:
for c in raw.columns:
    raw[c] = raw[c].str.strip().replace('', pd.NA)
profile_rows = []
for city, frame in raw.groupby('city', sort=True):
    for c in used:
        present = c in headers[city]
        series = frame[c] if c in frame else pd.Series(pd.NA, index=frame.index, dtype='string')
        profile_rows.append({'city': city, 'column': c, 'present': present, 'source_type': 'CSV text',
                             'analysis_type': 'Float64' if c in NUMERIC or c in MONETARY else 'string',
                             'non_null': int(series.notna().sum()), 'missing_pct': round(100 * series.isna().mean(), 2)})
profile = pd.DataFrame(profile_rows)
show(profile)
grain_rows = []
for city, f in raw.groupby('city'):
    grain_rows.append({'city': city, 'rows': len(f), 'missing_id': int(f['id'].isna().sum()),
        'duplicate_key_rows': int(f.duplicated(['id', 'snapshot_date'], keep=False).sum()),
        'scrape_date_differs_from_snapshot_label': int((f['last_scraped'].notna() & f['last_scraped'].ne(f['snapshot_date'])).sum())})
grain = pd.DataFrame(grain_rows)
show(grain)
pattern_columns = [c for c in ['price','bathrooms','bedrooms','beds','review_scores_rating','reviews_per_month','neighbourhood_group_cleansed'] if c in raw]
patterns = []
for city, f in raw.groupby('city'):
    labels = f[pattern_columns].isna().apply(lambda r: ', '.join(r.index[r]) or 'complete', axis=1)
    counts = labels.value_counts().rename_axis('missing_columns').reset_index(name='rows')
    counts['city'] = city
    counts['pct'] = (100 * counts['rows'] / len(f)).round(2)
    patterns.append(counts)
patterns = pd.concat(patterns, ignore_index=True)
show(patterns.groupby('city').head(10))

scrape_dates = raw.groupby(['city','last_scraped'],dropna=False).size().reset_index(name='rows')
show(scrape_dates)


## Parsing, price quality and extreme values

GBP context is established by the source region; the publisher documents the dollar sign as an export artifact ([data dictionary](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit)). Parsing accepts optional `£`, `$` or `GBP`, an optional sign, standard comma thousands groups and at most two decimal places. Malformed values remain missing in the parsed field and receive a separate failure flag. Currency provenance must be confirmed before treating any unexpected currency as GBP.

Zero, negative and missing targets are not silently removed. Prices below £10 or above £2,000 per night are review flags, not verified errors. A city-specific upper Tukey fence (Q3 + 1.5 × IQR) is descriptive only. Data-derived thresholds must be fitted on training data if adopted for model preprocessing. All positive parsed prices, including flagged extremes, remain in descriptive summaries.


In [ ]:
def parse_money(series):
    text = series.astype('string').str.strip()
    valid = text.str.fullmatch(r'(?:(?:£|\$|GBP)\s*)?[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d{1,2})?', na=False)
    numeric = text.where(valid).str.replace(r'^(?:£|\$|GBP)\s*', '', regex=True).str.replace(',', '', regex=False)
    return pd.to_numeric(numeric, errors='coerce').astype('Float64'), text.notna() & ~valid

analysis = raw.copy()
conversion_rows = []
for c in NUMERIC:
    if c in raw:
        analysis[c] = pd.to_numeric(raw[c], errors='coerce').astype('Float64')
        for city, idx in raw.groupby('city').groups.items():
            conversion_rows.append({'city': city, 'column': c, 'parse_failures': int((raw.loc[idx,c].notna() & analysis.loc[idx,c].isna()).sum())})
for c in MONETARY:
    if c in raw:
        analysis[c + '_gbp'], analysis[c + '_parse_failure'] = parse_money(raw[c])
price = analysis['price_gbp']
flags = pd.DataFrame(index=analysis.index)
flags['missing_price'] = raw['price'].isna()
flags['malformed_price'] = analysis['price_parse_failure']
flags['zero_price'] = price.eq(0).fillna(False)
flags['negative_price'] = price.lt(0).fillna(False)
flags['price_below_10'] = (price.gt(0) & price.lt(10)).fillna(False)
flags['price_above_2000'] = price.gt(2000).fillna(False)
flags['missing_id'] = raw['id'].isna()
flags['duplicate_key'] = raw.duplicated(['city','id','snapshot_date'], keep=False)
flags['nonpositive_or_fractional_accommodates'] = (analysis['accommodates'].le(0) | analysis['accommodates'].mod(1).ne(0)).fillna(False)
for c, limit in [('availability_30',30),('availability_60',60),('availability_90',90),('availability_365',365)]:
    if c in analysis:
        flags[c + '_invalid'] = (analysis[c].lt(0) | analysis[c].gt(limit) | analysis[c].mod(1).ne(0)).fillna(False)
flags['availability_nonmonotonic'] = False
for a,b in [('availability_30','availability_60'),('availability_60','availability_90'),('availability_90','availability_365')]:
    if a in analysis and b in analysis:
        flags['availability_nonmonotonic'] |= analysis[a].gt(analysis[b]).fillna(False)
for c in ['bathrooms','bedrooms','beds','number_of_reviews']:
    if c in analysis:
        flags[c + '_negative'] = analysis[c].lt(0).fillna(False)
if 'review_scores_rating' in analysis:
    flags['rating_outside_0_5'] = (analysis['review_scores_rating'].lt(0) | analysis['review_scores_rating'].gt(5)).fillna(False)
quality = flags.assign(city=analysis['city']).groupby('city').sum().reset_index()
show(quality)
conversions = pd.DataFrame(conversion_rows)
show(conversions[conversions['parse_failures'] > 0])
positive = analysis.loc[price.gt(0).fillna(False)].copy()
summary = positive.groupby('city')['price_gbp'].agg(['count','mean','median','min','max']).reset_index()
quantiles = positive.groupby('city')['price_gbp'].quantile([.25,.75,.95,.99]).unstack().reset_index()
quantiles.columns = ['city','p25','p75','p95','p99']
summary = summary.merge(quantiles,on='city')
summary['upper_iqr_fence'] = summary['p75'] + 1.5 * (summary['p75'] - summary['p25'])
fences = summary.set_index('city')['upper_iqr_fence']
analysis['price_upper_iqr_flag'] = price.gt(analysis['city'].map(fences)).fillna(False)
summary['above_iqr_rows'] = summary['city'].map(analysis.groupby('city')['price_upper_iqr_flag'].sum())
show(summary.round(2))
review_pattern = analysis.assign(has_reviews=analysis['number_of_reviews'].gt(0).fillna(False),
                                 missing_rating=analysis['review_scores_rating'].isna()).groupby(['city','has_reviews']).agg(
                                     rows=('id','size'), missing_rating_pct=('missing_rating', lambda s: 100*s.mean())).reset_index()
show(review_pattern.round(2))

price_missingness = analysis.assign(
    missing_price=analysis['price_gbp'].isna(),
    availability_group=analysis['has_availability'].fillna('missing'),
    zero_available_days=analysis['availability_365'].eq(0).fillna(False)
).groupby(['city','availability_group','zero_available_days'],dropna=False).agg(
    rows=('id','size'),missing_price_pct=('missing_price',lambda x:100*x.mean())).reset_index()
show(price_missingness.round(2))
bathroom_text = raw['bathrooms_text']
bathroom_count = pd.to_numeric(bathroom_text.str.extract(r'^(\d+(?:\.\d+)?)\s+(?:shared\s+|private\s+)?baths?$', expand=False),errors='coerce').astype('Float64')
half_bath = bathroom_text.str.fullmatch(r'(?i)(?:shared |private )?half-bath',na=False)
bathroom_count = bathroom_count.mask(half_bath,0.5)
bathroom_audit = pd.DataFrame({'city':raw['city'],'numeric_missing':analysis['bathrooms'].isna(),
    'text_present':bathroom_text.notna(),'recoverable_from_text':bathroom_count.notna(),
    'unrecognised_text':bathroom_text.notna() & bathroom_count.isna()}).groupby('city').sum().reset_index()
show(bathroom_audit)


## City and listing-segment comparisons

Price statistics use positive parsed targets. Every comparison includes its row count; missing segment labels remain visible. Availability is the number of calendar days marked available, not occupancy or realised demand. Full-range logarithmic price histograms retain extreme values. Neighbourhood comparisons use the ten largest groups per city to reduce unstable small-group rankings; they remain descriptive.


In [ ]:
segment_tables = {}
for segment in ['room_type','accommodates','neighbourhood_cleansed']:
    grouped = positive.groupby(['city',segment], dropna=False)['price_gbp'].agg(['size','median','mean']).reset_index()
    grouped.columns = ['city',segment,'priced_rows','median_gbp','mean_gbp']
    if segment == 'neighbourhood_cleansed':
        grouped = grouped.sort_values(['city','priced_rows'],ascending=[True,False]).groupby('city').head(10)
    segment_tables[segment] = grouped
    show(grouped.round(2))
room_mix = analysis.groupby(['city','room_type'],dropna=False).size().reset_index(name='rows')
room_mix['pct'] = (100 * room_mix['rows'] / room_mix.groupby('city')['rows'].transform('sum')).round(2)
show(room_mix)
availability = analysis.groupby('city')['availability_365'].agg(['count','mean','median','min','max']).reset_index()
show(availability.round(2))
review_summary = analysis.groupby('city')['review_scores_rating'].agg(['count','mean','median']).reset_index()
show(review_summary.round(2))
fig, axes = plt.subplots(1,2,figsize=(12,4),layout='constrained')
for city,f in positive.groupby('city'):
    axes[0].hist(np.log10(f['price_gbp'].astype(float)),bins=60,alpha=.5,label=f'{city.title()} (n={len(f):,})',density=True)
for city,f in analysis.groupby('city'):
    axes[1].hist(f['availability_365'].dropna().astype(float),bins=30,alpha=.5,label=city.title(),density=True)
axes[0].set(xlabel='log10(nightly advertised price / GBP)',ylabel='Density',title='Full positive price range')
axes[1].set(xlabel='Available days in the next 365 days',ylabel='Density',title='Calendar availability')
for ax in axes: ax.legend()
fig.savefig(FIGURE_ROOT/'price_availability.png',bbox_inches='tight')
plt.show()
fig, axes = plt.subplots(1,2,figsize=(12,4),layout='constrained')
for ax,city in zip(axes,['bristol','london']):
    f = segment_tables['room_type'].query('city == @city')
    ax.barh(f['room_type'].fillna('Missing'),f['median_gbp'].astype(float))
    ax.set(xlabel='Median nightly advertised price (GBP)',title=city.title(),
           xlim=(0, float(segment_tables['room_type']['median_gbp'].max()) * 1.15))
    for i, row in enumerate(f.itertuples(index=False)):
        ax.text(float(row.median_gbp) + 2, i, f'n={row.priced_rows:,}', va='center', fontsize=9)
fig.savefig(FIGURE_ROOT/'room_type_prices.png',bbox_inches='tight')
plt.show()


## Cleaning specification and leakage review

Recommended Silver behavior: retain every source row with provenance and quality flags. Quarantine missing or duplicate keys for resolution. Parse typed fields without coercing failures to valid values. Leave missing targets and attributes missing; do not replace prices with zero. Route nonpositive or malformed targets out of supervised evaluation while retaining them for inference where predictors are usable. Manually review extreme prices; do not impose EDA-derived caps on test data. Validate availability bounds and monotonicity, capacities and review scales. Preserve valid zero bedroom counts for studios.

Person 3 must reserve a per-city test split before model selection. Imputation, category vocabularies, outlier thresholds, feature selection and room-type/accommodates baseline medians must be learned on training rows only. EDA is descriptive across A/B; it is not a model-selection score. Consider grouping by host during splitting to assess dependence, but do not export host identifiers to the repository.

Prediction context: estimating the current advertised nightly price for an existing listing from non-price listing attributes available at the same snapshot. A conservative starting feature set is `room_type`, `property_type`, `accommodates`, `bedrooms`, `beds`, `bathrooms`, `latitude`, `longitude`, `minimum_nights` and `maximum_nights`, with explicit missing-value handling. `instant_bookable` and `neighbourhood_group_cleansed` are entirely missing in both pinned snapshots and should be omitted from the initial model. `city` and `neighbourhood_cleansed` are optional categorical features only with an unseen-category policy that works before City C arrives. Review and host-status attributes are conditional candidates for existing listings; they are inappropriate for cold-start listings without history. Exclude `review_scores_value` from the initial model because it reflects perceived value relative to price.

Explicitly exclude target `price`, every `price_quote_*` field (including dates and raw payloads), `estimated_revenue_l365d`, all other revenue/monetary/price-derived fields, IDs, names, URLs, free-text descriptions and scrape metadata. Estimated occupancy and calendar availability are excluded from the conservative feature set because they can encode demand responses to current price and their estimation assumptions require verification. Revenue divided by estimated occupancy can reconstruct a nightly price proxy; it must never become a feature. The audit below reports aggregate relationships for diagnosis only.


In [ ]:
leakage_rows = []
for city, header in headers.items():
    for c in header:
        if any(t in c.lower() for t in ['price','revenue','occupancy']) or c == 'review_scores_value':
            reason = ('Target' if c == 'price' else
                      'Price quotation or quotation metadata' if c.startswith('price_quote_') else
                      'Monetary estimate may directly encode target price' if 'revenue' in c else
                      'Estimation assumptions and demand endogeneity' if 'occupancy' in c else
                      'Value judgement depends on advertised price')
            leakage_rows.append({'city':city,'column':c,'decision':'exclude','reason':reason})
leakage = pd.DataFrame(leakage_rows)
show(leakage)
monetary_audit = []
for city,f in analysis.groupby('city'):
    for c in MONETARY:
        if c == 'price' or c + '_gbp' not in f: continue
        paired = f[['price_gbp', c+'_gbp']].dropna().astype(float)
        monetary_audit.append({'city':city,'column':c,'paired_rows':len(paired),
            'pearson_with_price': paired.corr().iloc[0,1] if len(paired)>1 else np.nan,
            'equal_to_price_rows':int(paired.iloc[:,0].eq(paired.iloc[:,1]).sum())})
monetary_audit = pd.DataFrame(monetary_audit)
show(monetary_audit.round(4))
rule_rows = []
for city,f in analysis.groupby('city'):
    idx=f.index
    for flag in flags:
        n=int(flags.loc[idx,flag].sum())
        rule_rows.append({'city':city,'rule':flag,'affected_rows':n,'pct':round(100*n/len(f),2)})
    rule_rows.append({'city':city,'rule':'upper_iqr_review_only','affected_rows':int(f['price_upper_iqr_flag'].sum()),'pct':round(100*f['price_upper_iqr_flag'].mean(),2)})
rule_impacts = pd.DataFrame(rule_rows)
show(rule_impacts)

revenue_proxy_audit = []
for city,f in analysis.groupby('city'):
    occupancy = f['estimated_occupancy_l365d']
    ratio = f['estimated_revenue_l365d_gbp'].div(occupancy.where(occupancy.gt(0)))
    pair = pd.DataFrame({'price':f['price_gbp'],'revenue_per_occupied_day':ratio}).dropna().astype(float)
    revenue_proxy_audit.append({'city':city,'paired_rows':len(pair),
        'pearson_with_price':pair.corr().iloc[0,1] if len(pair)>1 else np.nan,
        'equal_within_1p_rows':int(np.isclose(pair.iloc[:,0],pair.iloc[:,1],atol=.01,rtol=0).sum())})
revenue_proxy_audit = pd.DataFrame(revenue_proxy_audit)
show(revenue_proxy_audit.round(4))


## Reproducible aggregate report and handoff

The report and figures contain aggregates only. The source notebook retains no embedded outputs. Databricks writes reports to the configured artifacts volume; copy reviewed aggregate artifacts to `docs/` for version control. Optional column absence, schema changes and failed checks must be resolved before the production pipeline is frozen. ETL and ML owners approve their interfaces independently; this analysis does not mark their work complete.


In [ ]:
report = [
    '# London and Bristol: EDA findings',
    f"Execution completed: {datetime.now(timezone.utc).isoformat()}. Python {platform.python_version()}; pandas {pd.__version__}; NumPy {np.__version__}.",
    '## Findings and limitations',
    f"London has {analysis['city'].eq('london').sum():,} rows and Bristol {analysis['city'].eq('bristol').sum():,}; median positive prices are £{summary.set_index('city').loc['london','median']:.2f} and £{summary.set_index('city').loc['bristol','median']:.2f}.",
    'The tables below describe all A/B snapshot rows and positive parsed prices. No raw rows were removed or overwritten. Prices are advertised nightly amounts in GBP, not realised booking revenue. City snapshots differ by five days. Model error and transfer performance remain unmeasured.',
    '## Dataset provenance and grain', table(provenance), table(grain), table(scrape_dates), 'Published snapshot labels and actual listing scrape dates differ. Preserve both; do not treat the difference alone as a corrupt record.',
    '## Price distribution (GBP)', table(summary.round(2)),
    'Positive parsed targets form the denominator for price statistics. Quality flags below cover all source rows.',
    '## Quality flags and cleaning impact', table(rule_impacts),
    'Flag counts overlap and must not be summed into a deletion count. Values below £10 or above £2,000 and upper-IQR flags require review rather than automatic exclusion.',
    '## Column profile', 'The complete used-column schema and missingness audit is in [eda_profile.md](eda_profile.md).',
    '## Missingness patterns', table(patterns.groupby('city').head(5)),
    'Pattern percentages use all rows in each city. Complete rows are included; combinations identify jointly absent fields.',
    '## Review-history missingness', table(review_pattern.round(2)), 'Missing ratings among rows without reviews are structurally plausible; unexplained gaps among reviewed listings require checking. Do not fabricate ratings.', '## Price missingness by availability',table(price_missingness.round(2)), 'This association does not identify the cause of missing prices. Do not infer occupancy from unavailable days or impute target prices from quote fields.', '## Bathroom recovery audit',table(bathroom_audit),
    '## Numeric parse failures', table(conversions[conversions.parse_failures > 0]),
    '## City comparisons', '### Calendar availability in the next 365 days (days)', table(availability.round(2)), '### Review rating (0–5)', table(review_summary.round(2)), '### Room-type composition', table(room_mix),
    'Availability is calendar availability, not occupancy. Review aggregates exclude missing scores and report their non-null denominator.',
    '## Segment prices (GBP)',
]
for segment,f in segment_tables.items():
    report.extend([f'### {segment}',table(f.round(2))])
report.extend([
    '## Leakage audit', table(leakage.drop(columns='city').drop_duplicates()), table(monetary_audit.round(4)), table(revenue_proxy_audit.round(4)),
    'Exclude all price quotations, quotation metadata, revenue estimates, perceived-value scores and price-derived features. Correlation is diagnostic and is not evidence that an unflagged field is safe. Revenue/occupancy ratios are prohibited features.',
    '## Verified leakage findings', 'The per-night quotation equals the target on every paired row in both cities. Revenue divided by estimated occupied days is also an almost exact price proxy. Neither belongs in the predictor set.',
    '## ETL recommendations',
    'Retain immutable Bronze files and provenance. In Silver, preserve rows with quality flags, quarantine invalid keys, parse GBP explicitly, retain missing values, and distinguish malformed from absent values. Exclude invalid targets only from supervised targets; keep their records for inference where possible. Review extremes without automatic capping. Validate capacity, availability and score domains. See cleaning_specification.md for rule rationales.',
    '## ML handoff',
    'Begin with room_type, property_type, accommodates, bedrooms, beds, bathrooms, latitude, longitude, minimum_nights and maximum_nights. Omit instant_bookable and neighbourhood_group_cleansed because both are entirely missing in the pinned snapshots; derive bathroom counts from text. Missing attributes need train-only imputation and indicators. City and neighbourhood require a pre-specified unseen-category policy. Reserve a test set per city before model selection; fit preprocessing, thresholds and baseline medians on training data only. No EDA summary is a test-set model metric.',
    '## Unresolved questions',
    'The publisher confirms that price uses local currency and that the dollar sign is an export artifact. Confirm newer quotation and revenue-estimation semantics against publisher metadata. Resolve structural missingness before selecting optional features. Agree table contracts and shared workspace access. ETL execution, price model, dashboard, Genie and City C transfer are separate deliverables.',
    '## Figures',
    '![Full price range and availability](figures/price_availability.png)',
    '![Median advertised price by room type](figures/room_type_prices.png)',
    '## Source',
    '[Inside Airbnb data downloads](https://insideairbnb.com/get-the-data/), CC BY 4.0. Snapshot URLs and checksums are recorded in sources.json.',
])
report_text = '\n\n'.join(report) + '\n'
(OUTPUT_ROOT / 'eda_findings.md').write_text(report_text)
(OUTPUT_ROOT / 'eda_profile.md').write_text('# Used-column profile\n\n' + table(profile) + '\n\n## All missingness patterns\n\n' + table(patterns) + '\n')
(OUTPUT_ROOT / 'eda_run.json').write_text(json.dumps({
    'completed_utc': datetime.now(timezone.utc).isoformat(), 'python': platform.python_version(),
    'pandas': pd.__version__, 'numpy': np.__version__, 'sources':sources,
    'rows': {str(k):int(v) for k,v in analysis.groupby('city').size().items()},
    'raw_rows_removed':0, 'analysis_scope':['london','bristol']
},indent=2)+'\n')
display(Markdown('## Verified execution summary'))
show(provenance[['city','snapshot_date','rows']])
show(summary[['city','count','median','p95']].round(2))
print(f'Aggregate report: {OUTPUT_ROOT / "eda_findings.md"}')
